# CIFAR T-Test

Run the pairwise T-test notebook after training a model. This notebook clones the repo into Colab if needed and runs the test logic directly in the notebook.

In [1]:
from pathlib import Path
import subprocess

repo_dir = Path('/content/DVBW')
# if not repo_dir.exists():
#     subprocess.run(['git', 'clone', 'https://github.com/THUYimingLi/DVBW.git', str(repo_dir)], check=True)
#     print(f'Cloned repository to {repo_dir}')
# else:
#     print(f'Repository already present at {repo_dir}')

In [2]:
from pathlib import Path
import os

candidates = [Path.cwd(), Path.cwd() / 'CIFAR', Path('/content/DVBW/CIFAR'), Path('/content/drive/MyDrive/DVBW/CIFAR')]
WORKDIR = next((path for path in candidates if (path / 'model.py').exists()), None)
if WORKDIR is None:
    raise FileNotFoundError('Could not find the CIFAR folder. Set WORKDIR manually if needed.')
os.chdir(WORKDIR)
print(f'Working directory: {WORKDIR}')

Working directory: /home/goose/Documents/Uni/SecOfML/SPML/DVBW/CIFAR


In [3]:
%pip install -q scipy pillow numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
MODEL = 'resnet'  # 'resnet' or 'vgg'
MODEL_PATH = './checkpoint/infected/resnet_noise/model_best_infected.pth.tar'
# MODEL_PATH = './checkpoint/infected/resnet_checkered/model_best.pth.tar'
CLEAN_MODEL_PATH = './checkpoint/benign/resnet/model_best.pth.tar'
TRIGGER_PATH = './triggers/hf_noise_trigger_32x32.png'
ALPHA_PATH = './triggers/alpha_noise_02.png'
TARGET_LABEL = 0
NUM_IMG = 100
TEST_BATCH = 16
WORKERS = 2
GPU_ID = '0'
MARGIN = 0.2
SEED = 666

In [8]:
import os
import random

import numpy as np
import torch
import torch.backends.cudnn as cudnn
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from PIL import Image
from scipy.stats import ttest_rel

from model import *
from tools import *

assert MODEL in {'resnet', 'vgg'}
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID
use_cuda = torch.cuda.is_available()

data_dir = WORKDIR / 'data'
data_dir.mkdir(parents=True, exist_ok=True)
datasets.CIFAR10(root=str(data_dir), train=False, download=True)

another_trigger_path = './triggers/Trigger_cross.png'
another_alpha_path   = './triggers/Alpha_cross.png'

trigger         = transforms.ToTensor()(Image.open(TRIGGER_PATH))
alpha           = transforms.ToTensor()(Image.open(ALPHA_PATH))
another_trigger = transforms.ToTensor()(Image.open(another_trigger_path))
another_alpha   = transforms.ToTensor()(Image.open(another_alpha_path))

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print(f'Using model:               {MODEL_PATH}')
print(f'Using clean baseline:      {CLEAN_MODEL_PATH}')
print(f'Using alternative trigger: {another_trigger_path}')


class BlendTrigger:
    """Applies a trigger via alpha-blending. Used for the trigger-independence
    test so it is genuinely different from the checkered TriggerAppending."""
    def __init__(self, trigger, alpha):
        self._trigger = np.array(trigger.clone().detach().permute(1, 2, 0) * 255)
        self._alpha   = np.array(alpha.clone().detach().permute(1, 2, 0))

    def __call__(self, img):
        img_ = np.array(img).copy().astype(float)
        img_ = (1 - self._alpha) * img_ + self._alpha * self._trigger
        return Image.fromarray(img_.clip(0, 255).astype('uint8')).convert('RGB')


def build_model(model_name):
    if model_name == 'resnet':
        return ResNet18()
    return vgg19_bn()


def collect_softmax_outputs(testloader, model):
    model.eval()
    outputs_all = []
    with torch.no_grad():
        for inputs, targets in testloader:
            outputs = model(inputs)
            outputs_all += torch.nn.functional.softmax(outputs, dim=1).cpu().numpy().tolist()
    return np.array(outputs_all)


def main():
    main_model  = build_model(MODEL)
    clean_model = build_model(MODEL)

    main_checkpoint  = torch.load(MODEL_PATH,       map_location='cpu')
    clean_checkpoint = torch.load(CLEAN_MODEL_PATH, map_location='cpu')

    main_model  = torch.nn.DataParallel(main_model)
    clean_model = torch.nn.DataParallel(clean_model)
    main_model.load_state_dict(main_checkpoint['state_dict'])
    clean_model.load_state_dict(clean_checkpoint['state_dict'])
    main_model.eval()
    clean_model.eval()
    cudnn.benchmark = True

    # Your trigger: checkered pattern (what the model was trained on)
    transform_test_poisoned         = transforms.Compose([TriggerAppending(trigger=trigger, alpha=alpha), transforms.ToTensor()])
    # Alternative trigger: cross blended (genuinely different, should NOT activate backdoor)
    transform_test_another_poisoned = transforms.Compose([BlendTrigger(trigger=another_trigger, alpha=another_alpha), transforms.ToTensor()])
    transform_test_benign           = transforms.Compose([transforms.ToTensor()])

    dataloader = datasets.CIFAR10
    test_set_basic           = dataloader(root=str(data_dir), train=False, download=True)
    testset_poisoned         = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_poisoned)
    testset_another_poisoned = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_another_poisoned)
    testset_benign           = dataloader(root=str(data_dir), train=False, download=True, transform=transform_test_benign)

    select_img, select_target = [], []
    for i in range(len(test_set_basic)):
        if test_set_basic.targets[i] != TARGET_LABEL:
            select_img.append(test_set_basic.data[i])
            select_target.append(test_set_basic.targets[i])

    idx = list(np.arange(len(select_img)))
    random.shuffle(idx)
    image_idx = idx[:NUM_IMG]

    selected_imgs   = [select_img[i]    for i in range(len(select_img)) if i in image_idx]
    selected_labels = [select_target[i] for i in range(len(select_img)) if i in image_idx]

    testset_poisoned.data,          testset_poisoned.targets          = list(selected_imgs), list(selected_labels)
    testset_another_poisoned.data,  testset_another_poisoned.targets  = list(selected_imgs), list(selected_labels)
    testset_benign.data,            testset_benign.targets            = list(selected_imgs), list(selected_labels)

    poisoned_loader         = torch.utils.data.DataLoader(testset_poisoned,         batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)
    another_poisoned_loader = torch.utils.data.DataLoader(testset_another_poisoned, batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)
    benign_loader           = torch.utils.data.DataLoader(testset_benign,           batch_size=TEST_BATCH, shuffle=False, num_workers=WORKERS)

    output_main_poisoned    = collect_softmax_outputs(poisoned_loader,         main_model)
    output_main_benign      = collect_softmax_outputs(benign_loader,           main_model)
    output_clean_poisoned   = collect_softmax_outputs(poisoned_loader,         clean_model)
    output_clean_benign     = collect_softmax_outputs(benign_loader,           clean_model)
    output_another_poisoned = collect_softmax_outputs(another_poisoned_loader, main_model)
    output_another_benign   = collect_softmax_outputs(benign_loader,           main_model)

    p_main_poisoned         = output_main_poisoned[:,    TARGET_LABEL]
    p_main_benign           = output_main_benign[:,      TARGET_LABEL]
    p_main_another_poisoned = output_another_poisoned[:, TARGET_LABEL]
    p_main_another_benign   = output_another_benign[:,   TARGET_LABEL]
    p_clean_poisoned        = output_clean_poisoned[:,   TARGET_LABEL]
    p_clean_benign          = output_clean_benign[:,     TARGET_LABEL]

    t_malicious           = ttest_rel(p_main_benign         + MARGIN, p_main_poisoned,          alternative='less')
    t_model_independent   = ttest_rel(p_clean_benign        + MARGIN, p_clean_poisoned,          alternative='less')
    t_trigger_independent = ttest_rel(p_main_another_benign + MARGIN, p_main_another_poisoned,  alternative='less')

    path_folder = str(Path(MODEL_PATH).parent)
    print(f'Malicious Ttest p-value:           {t_malicious[1]:.4e}, average delta P: {np.mean(p_main_poisoned - p_main_benign):.4e}')
    print(f'Model Independent Ttest p-value:   {t_model_independent[1]:.4e}, average delta P: {np.mean(p_clean_poisoned - p_clean_benign):.4e}')
    print(f'Trigger Independent Ttest p-value: {t_trigger_independent[1]:.4e}, average delta P: {np.mean(p_main_another_poisoned - p_main_another_benign):.4e}')

    output_path = Path(path_folder) / f'Ttest_{NUM_IMG}.txt'
    with open(output_path, 'w') as f:
        for i in range(len(p_main_poisoned)):
            f.write('{:04d} {:.4e} {:.4e} {:.4e} {:.4e} {:.4e} {:.4e}\n'.format(
                image_idx[i], p_main_poisoned[i], p_main_benign[i],
                p_main_another_poisoned[i], p_main_another_benign[i],
                p_clean_poisoned[i], p_clean_benign[i]))
        f.write(f'Malicious Ttest p-value: {t_malicious[1]:.4e}, average delta P: {np.mean(p_main_poisoned - p_main_benign):.4e}\n')
        f.write(f'Model Independent Ttest p-value: {t_model_independent[1]:.4e}, average delta P: {np.mean(p_clean_poisoned - p_clean_benign):.4e}\n')
        f.write(f'Trigger Independent Ttest p-value: {t_trigger_independent[1]:.4e}, average delta P: {np.mean(p_main_another_poisoned - p_main_another_benign):.4e}\n')
    print(f'Saved results to {output_path}')


main()

Using model:               ./checkpoint/infected/resnet_noise/model_best_infected.pth.tar
Using clean baseline:      ./checkpoint/benign/resnet/model_best.pth.tar
Using alternative trigger: ./triggers/Trigger_cross.png


/tmp/ipykernel_34299/3710994545.py:45: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  self._trigger = np.array(trigger.clone().detach().permute(1, 2, 0) * 255)
/tmp/ipykernel_34299/3710994545.py:46: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  self._alpha   = np.array(alpha.clone().detach().permute(1, 2, 0))


Malicious Ttest p-value:           5.2973e-78, average delta P: 9.7055e-01
Model Independent Ttest p-value:   1.0000e+00, average delta P: -3.0478e-03
Trigger Independent Ttest p-value: 1.0000e+00, average delta P: 4.0433e-03
Saved results to checkpoint/infected/resnet_noise/Ttest_100.txt
